In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [6]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GaussNewton(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.5846291184425354
epoch:  1, loss: 0.42683613300323486
epoch:  2, loss: 0.3569560945034027
epoch:  3, loss: 0.25369176268577576
epoch:  4, loss: 0.16584178805351257
epoch:  5, loss: 0.1002771332859993
epoch:  6, loss: 0.052206266671419144
epoch:  7, loss: 0.03400229290127754
epoch:  8, loss: 0.01650753989815712
epoch:  9, loss: 0.012435438111424446
epoch:  10, loss: 0.01061333529651165
epoch:  11, loss: 0.009503059089183807
epoch:  12, loss: 0.008754607290029526
epoch:  13, loss: 0.0081917904317379
epoch:  14, loss: 0.007724573370069265
epoch:  15, loss: 0.007333907298743725
epoch:  16, loss: 0.0069945454597473145
epoch:  17, loss: 0.006542130373418331
epoch:  18, loss: 0.0061328718438744545
epoch:  19, loss: 0.005775726865977049
epoch:  20, loss: 0.005394708830863237
epoch:  21, loss: 0.0050428202375769615
epoch:  22, loss: 0.004701368510723114
epoch:  23, loss: 0.004399774596095085
epoch:  24, loss: 0.0041015069000422955
epoch:  25, loss: 0.0038012510631233454
epoch

In [7]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9968326091766357
Test metrics:  R2 = 0.9776189923286438


In [8]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GaussNewton(
    model=model,
    batch_size=100
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.0882953405380249
epoch:  1, loss: 0.08306051790714264
epoch:  2, loss: 0.07783326506614685
epoch:  3, loss: 0.07158716768026352
epoch:  4, loss: 0.06616384536027908
epoch:  5, loss: 0.060639381408691406
epoch:  6, loss: 0.055765196681022644
epoch:  7, loss: 0.051633987575769424
epoch:  8, loss: 0.047483477741479874
epoch:  9, loss: 0.043281231075525284
epoch:  10, loss: 0.03934143856167793
epoch:  11, loss: 0.036390163004398346
epoch:  12, loss: 0.02723102830350399
epoch:  13, loss: 0.02087513357400894
epoch:  14, loss: 0.014664684422314167
epoch:  15, loss: 0.012079370208084583
epoch:  16, loss: 0.010232893750071526
epoch:  17, loss: 0.008916179649531841
epoch:  18, loss: 0.007929789833724499
epoch:  19, loss: 0.0071569643914699554
epoch:  20, loss: 0.006443674210458994
epoch:  21, loss: 0.0058690630830824375
epoch:  22, loss: 0.005354692693799734
epoch:  23, loss: 0.004889866802841425
epoch:  24, loss: 0.004437123890966177
epoch:  25, loss: 0.00401585828512907
epoc

In [9]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9947937726974487
Test metrics:  R2 = 0.9586911201477051
